<a href="https://colab.research.google.com/github/DanielYaari28/EDM-Music-Search-Engine/blob/main/EDM_search_starter_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

*This is a starter notebook. This is designed to give you a jumping off point only. *

Make sure to
`pip install chromadb gradio sentence-transformers`


In [1]:
# Load an embedding model (you can use any embedding model)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
!pip install datasets

In [3]:
# Provide your own dataset here. Decide on a chunking strategy and implement here.

from datasets import load_dataset

ds = load_dataset("maharshipandya/spotify-tracks-dataset", split="train")
df = ds.to_pandas()

edm_genres = ["edm", "house", "deep-house", "progressive-house", "techno",
              "trance", "dubstep", "electro", "drum-and-bass", "electronic"]
df = df[df["track_genre"].isin(edm_genres)]
df = df.drop_duplicates(subset=["track_name", "artists"]).sample(2000, random_state=42)

def describe(row):
    energy = "high-energy" if row["energy"] > 0.7 else "mellow" if row["energy"] < 0.4 else "moderate-energy"
    dance  = "very danceable" if row["danceability"] > 0.7 else "less danceable"
    mood   = "upbeat and happy" if row["valence"] > 0.6 else "dark or moody" if row["valence"] < 0.3 else "neutral mood"
    vocals = "instrumental" if row["instrumentalness"] > 0.5 else "with vocals"
    return (f"{row['track_name']} by {row['artists']}. "
            f"A {energy}, {dance} {row['track_genre']} track, {mood}, {vocals}, "
            f"around {round(row['tempo'])} BPM.")

texts = df.apply(describe, axis=1).tolist()
print(texts[0])

Irresponsables - En Vivo by Babasónicos. A high-energy, less danceable electronic track, upbeat and happy, with vocals, around 188 BPM.


In [4]:
# Generate embeddings
embeddings = model.encode(texts)

In [7]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 123.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 122.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 51.9 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conv

In [13]:
import chromadb

client = chromadb.PersistentClient(path="chromadb")
collection = client.get_or_create_collection("edm_songs", metadata={"hnsw:space": "cosine"})

metadatas = df[["track_id", "track_name", "artists", "track_genre", "tempo"]].to_dict("records")

collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=[str(i) for i in range(len(texts))]
)
print(collection.count(), "songs saved")

2000 songs saved


In [14]:
print(chromadb.__version__)   # write this down

!zip -r chromadb.zip chromadb
from google.colab import files
files.download("chromadb.zip")

1.5.9
  adding: chromadb/ (stored 0%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/ (stored 0%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/index_metadata.pickle (deflated 63%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/link_lists.bin (deflated 84%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/data_level0.bin (deflated 12%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/header.bin (deflated 55%)
  adding: chromadb/f6e7361f-3666-4363-a375-cc8c34f2fe24/length.bin (deflated 85%)
  adding: chromadb/chroma.sqlite3 (deflated 41%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Define a search function
def semantic_search(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=1)
    return "\n\n".join(results['documents'][0])

In [16]:
import gradio as gr

GENRES = ["Any"] + sorted(df["track_genre"].unique().tolist())

def semantic_search(query, genre, n_results):
    if not query.strip():
        return "Type a vibe to search for 🎧"
    where = None if genre == "Any" else {"track_genre": genre}
    results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=int(n_results),
        where=where,
    )
    cards = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        match = round((1 - dist) * 100)
        link = f"https://open.spotify.com/track/{meta['track_id']}"
        cards.append(
            f"### {meta['track_name']}\n"
            f"**{meta['artists']}** · {meta['track_genre']} · {round(meta['tempo'])} BPM · "
            f"{match}% match · [Listen on Spotify]({link})"
        )
    return "\n\n---\n\n".join(cards) if cards else "No matches. Try another genre."

with gr.Blocks(theme=gr.themes.Soft(primary_hue="purple")) as demo:
    gr.Markdown("# EDM Music Search\nDescribe the vibe you want and find matching tracks.")
    with gr.Row():
        query = gr.Textbox(label="Describe a vibe", placeholder="techno, pop remix, dance", scale=3)
        genre = gr.Dropdown(GENRES, value="Any", label="Genre", scale=1)
    n_results = gr.Slider(1, 10, value=5, step=1, label="Number of results")
    search_btn = gr.Button("Search", variant="primary")
    output = gr.Markdown()

    gr.Examples(
        examples=["chill track to study to", "EDM festival sets",
                  "summer house", "instrumental techno"],
        inputs=query,
    )

    search_btn.click(semantic_search, [query, genre, n_results], output)
    query.submit(semantic_search, [query, genre, n_results], output)

demo.launch()

/tmp/ipykernel_1429/4180688800.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="purple")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76e6972137a252a0b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Next Steps:

- Save your chromadb
- Write code to load your chromadb
- Convert your loading, search function and app code to an app.py script
- Create a requirements.txt file
- Test your code!!
- Push your git repo to Hugging Face Spaces


___
Your file tree in your repository will look like this:

- chromadb/
  - chroma.sqlite3
- README.md
- requirements.txt
- app.py